# RAID v2 end-to-end Colab runner

This notebook mounts Google Drive, clones the `jp_branch` branch, installs dependencies, downloads the datasets needed for Phases A-E, and then runs the matrix phase by phase. All outputs are written under `RAID_ARTIFACT_ROOT` on Drive so the run is resumable after Colab disconnects.

In [ ]:
# === 1. Mount Drive and set artifact root ===
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

os.environ['RAID_ARTIFACT_ROOT'] = '/content/drive/MyDrive/raid_v2'
ARTIFACT_ROOT = Path(os.environ['RAID_ARTIFACT_ROOT'])
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print('RAID_ARTIFACT_ROOT =', ARTIFACT_ROOT)

In [ ]:
# === 2. Clone or refresh the repo ===
REPO_URL = 'https://github.com/ConstantinVictorBeatErtel/RAID.git'
REPO_DIR = Path('/content/RAID')
BRANCH = 'jp_branch'

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    %cd /content/RAID
    !git fetch origin
    !git checkout {BRANCH}
    !git reset --hard origin/{BRANCH}

%cd /content/RAID
!git status --short --branch
!ls v2 | head

In [ ]:
# === 3. Install Python dependencies ===
%cd /content/RAID
!pip install -q -r /content/RAID/v2/requirements.txt

In [ ]:
# === 4. Inline RoboMimic downloader ===
# Mirrors the Stanford CDN into the exact Drive layout v2 expects.
#
# Files are written as:
#   <artifact_root>/data/robomimic/v1.5/<task>/<variant>/<modality>_v141.hdf5
#
# This matches the Colab flow that produced the working Phase A/B/C runs.

import sys
import time
import urllib.request

ROBOMIMIC_BASE = 'http://downloads.cs.stanford.edu/downloads/rt_benchmark'
ROBOMIMIC_ROOT = ARTIFACT_ROOT / 'data' / 'robomimic' / 'v1.5'

def _progress(block_num, block_size, total_size):
    if total_size <= 0:
        return
    pct = min(100, 100 * block_num * block_size / total_size)
    sys.stdout.write(f'\r  {pct:5.1f}%  ({block_num * block_size / 1e6:.1f} / {total_size / 1e6:.1f} MB)')
    sys.stdout.flush()

def fetch_robomimic(task: str, variant: str, modality: str = 'low_dim') -> Path:
    """Download one RoboMimic HDF5 to Drive at the layout v2 expects.

    Files are stored as <task>/<variant>/<modality>_v141.hdf5 so the
    v2 hdf5_path_for resolver finds them on the first candidate path.
    """
    src_name = 'low_dim.hdf5' if modality == 'low_dim' else 'image.hdf5'
    dst = ROBOMIMIC_ROOT / task / variant / f'{modality}_v141.hdf5'
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 100_000:
        print(f'[skip ] {task}/{variant}/{modality}  ({dst.stat().st_size / 1e6:.1f} MB)')
        return dst
    url = f'{ROBOMIMIC_BASE}/{task}/{variant}/{src_name}'
    tmp = dst.with_suffix(dst.suffix + '.tmp')
    print(f'[fetch] {url}')
    t0 = time.time()
    try:
        urllib.request.urlretrieve(url, tmp, reporthook=_progress)
        sys.stdout.write('\n')
        tmp.replace(dst)
    except Exception as exc:
        if tmp.exists():
            tmp.unlink()
        raise RuntimeError(f'failed to download {url}: {exc}')
    print(f'[done ] {dst}  ({dst.stat().st_size / 1e6:.1f} MB in {time.time() - t0:.1f}s)')
    return dst

# Phase A only needs lift/ph low_dim.
# Phase B uses the low_dim files below.
# Phase C/D use square/ph image.
for task, variant, modality in [
    ('lift', 'ph', 'low_dim'),
    ('lift', 'mh', 'low_dim'),
    ('can', 'ph', 'low_dim'),
    ('can', 'mh', 'low_dim'),
    ('square', 'ph', 'low_dim'),
    ('square', 'mh', 'low_dim'),
    ('transport', 'ph', 'low_dim'),
    ('transport', 'mh', 'low_dim'),
    ('square', 'ph', 'image'),
]:
    fetch_robomimic(task, variant, modality)

In [ ]:
# === 5. Verify the files are really on Drive before training ===
from pathlib import Path

root = Path('/content/drive/MyDrive/raid_v2/data/robomimic/v1.5')
checks = [
    root / 'lift/ph/low_dim_v141.hdf5',
    root / 'lift/mh/low_dim_v141.hdf5',
    root / 'can/ph/low_dim_v141.hdf5',
    root / 'can/mh/low_dim_v141.hdf5',
    root / 'square/ph/low_dim_v141.hdf5',
    root / 'square/mh/low_dim_v141.hdf5',
    root / 'transport/ph/low_dim_v141.hdf5',
    root / 'transport/mh/low_dim_v141.hdf5',
    root / 'square/ph/image_v141.hdf5',
]

missing = []
for p in checks:
    ok = p.exists()
    print(p, ok, f'{p.stat().st_size/1e6:.1f} MB' if ok else 'MISSING')
    if not ok:
        missing.append(str(p))

if missing:
    raise RuntimeError('Missing RoboMimic files on Drive:\n' + '\n'.join(missing))


In [ ]:
# === 6. GPU sanity check ===
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Phase A

Reproduce the low-dim Lift baseline in the new harness.

In [ ]:
# === 6. Run Phase A ===
%cd /content/RAID
!python3 -m v2.run_matrix --phase A

## Phase B

Cross-dataset low-dim runs on the mixed RoboMimic tasks.

In [ ]:
# === 7. Refresh repo before later phases ===
%cd /content/RAID
!git fetch origin
!git checkout jp_branch
!git reset --hard origin/jp_branch

In [ ]:
# === 8. Run Phase B ===
%cd /content/RAID
!python3 -m v2.run_matrix --phase B

## Phase C

Cache DINOv2 image features for `robomimic_square_ph_image`, then run the image-feature matrix.

In [ ]:
# === 9. Extract DINOv2 features and run Phase C ===
%cd /content/RAID
!python3 -m v2.features --encoders dinov2 --datasets robomimic_square_ph_image
!python3 -m v2.run_matrix --phase C

## Phase D

Encoder ablation on the Theia features.

In [ ]:
# === 10. Extract Theia features and run Phase D ===
%cd /content/RAID
!python3 -m v2.features --encoders theia --datasets robomimic_square_ph_image
!python3 -m v2.run_matrix --phase D

## Phase E

Download LIBERO, cache DINOv2 features for `libero_goal`, and run the multi-task retrieval test.

In [ ]:
# === 11. Download LIBERO, extract features, and run Phase E ===
%cd /content/RAID
!python3 -m v2.runtime.data_download --include-libero
!python3 -m v2.features --encoders dinov2 --datasets libero_goal
!python3 -m v2.run_matrix --phase E

## Aggregate results

Build the forest plot from all completed runs. This script must be run as a module.

In [ ]:
# === 12. Aggregate completed runs into summary figures ===
%cd /content/RAID
!python3 -m v2.notebooks.03_matrix_results

In [ ]:
# === 13. Inspect generated figure paths ===
from pathlib import Path

fig_root = Path('/content/drive/MyDrive/raid_v2/results/figures')
for path in [fig_root / 'matrix_forest.png']:
    print(path, path.exists(), f'{path.stat().st_size / 1e6:.2f} MB' if path.exists() else '')

pred_root = fig_root / 'predictions'
if pred_root.exists():
    grids = sorted(pred_root.glob('*/grid.png'))
    print('prediction grids:', len(grids))
    for p in grids[-10:]:
        print(' ', p)

In [ ]:
# === 14. Optional: dry-run resume checks ===
%cd /content/RAID
!python3 -m v2.run_matrix --phase A --dry-run
!python3 -m v2.run_matrix --phase B --dry-run
!python3 -m v2.run_matrix --phase C --dry-run
!python3 -m v2.run_matrix --phase D --dry-run
!python3 -m v2.run_matrix --phase E --dry-run